<a href="https://colab.research.google.com/github/italobotelho/PI2026.1/blob/main/01_ETL_Extracao_SINAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 **Projeto SIEST: Sistema de Inteligência Epidemiológica e Socioespacial**
## Notebook 01: Extração e Transformação (Camada Bronze)

Este notebook é o motor de aquisição de dados do projeto. Ele é responsável por extrair os dados brutos de morbidade do SINAN (Sistema de Informação de Agravos de Notificação) através do FTP do DataSUS.

### 0. Configuração do Ambiente (Environment Setup)
Instalação das bibliotecas governamentais necessárias para a extração (PySUS).

In [ ]:
!pip install PySUS -q

### 1. Importação de Bibliotecas e Configurações Globais

In [ ]:
import pandas as pd
import os
import shutil
from pysus.ftp.databases.sinan import SINAN

# Configurações Globais do Pipeline
sinan = SINAN().load()
IBGE_CAMPINAS = 350950
anos_analise = range(2014, 2027)
doencas = ['DENG', 'CHIK', 'ZIKA', 'LEPT', 'HEPA']

# Dicionário de colunas de interesse para o Grafo
colunas_alvo = [
    'ID_MUNICIP', 'ID_UNIDADE', 'DT_NOTIFIC', 'DT_SIN_PRI',
    'NU_IDADE_N', 'CS_SEXO', 'HOSPITALIZ', 'CLASSI_FIN', 'EVOLUCAO', 'CRITERIO'
]

PASTA_BRONZE = 'dados_sinan_parquet'
os.makedirs(PASTA_BRONZE, exist_ok=True)

### 2. Processamento em Lotes (Chunking)

O pipeline abaixo itera sobre cada doença e ano. Para garantir alta performance e resiliência a falhas de tipagem nas bases históricas (ex: 2014), o filtro espacial aplica uma conversão vetorizada para *string* combinada com a função `.startswith()`. Arquivos temporários são deletados a cada iteração para preservar o disco virtual.

In [ ]:
print("Iniciando ETL com Processamento em Lotes (Anti-OOM e Alta Performance)...")

for doenca in doencas:
    print(f"\n--- Extraindo dados para: {doenca} ---")

    for ano in anos_analise:
        try:
            arquivos = sinan.get_files(doenca, ano)

            if not arquivos:
                print(f"[{ano}] Sem arquivos no servidor para {doenca}.")
                continue

            pasta_cache = sinan.download(arquivos)
            caminho_pasta = str(pasta_cache)
            chunks = [f for f in os.listdir(caminho_pasta) if f.endswith('.parquet')]
            lista_campinas = []

            for chunk in chunks:
                caminho_arquivo = os.path.join(caminho_pasta, chunk)
                df_chunk = pd.read_parquet(caminho_arquivo)

                # Filtro contra dados sujos
                if 'ID_MUNICIP' in df_chunk.columns:
                    mascara = df_chunk['ID_MUNICIP'].astype(str).str.startswith(str(IBGE_CAMPINAS), na=False)
                    df_filtrado = df_chunk[mascara].copy()
                else:
                    df_filtrado = pd.DataFrame()

                if not df_filtrado.empty:
                    colunas_existentes = df_filtrado.columns.intersection(colunas_alvo)
                    lista_campinas.append(df_filtrado[colunas_existentes])

                # Limpeza de memória RAM
                del df_chunk
                del df_filtrado

            if lista_campinas:
                df_campinas_final = pd.concat(lista_campinas, ignore_index=True)
                nome_arquivo = f"{PASTA_BRONZE}/{doenca}_campinas_{ano}.parquet"
                df_campinas_final.to_parquet(nome_arquivo, index=False, compression='snappy')

                print(f"[{ano}] Sucesso! {len(df_campinas_final)} casos salvos.")
                del df_campinas_final
            else:
                print(f"[{ano}] 0 casos registrados em Campinas para {doenca}.")

            # Limpeza do disco (Cache PySUS)
            shutil.rmtree(caminho_pasta, ignore_errors=True)

        except Exception as e:
            print(f"[{ano}] Erro ao processar {doenca}: {e}")

print("\nPipeline do SIEST concluído com sucesso!")